# GPU RAID — воркер на Google Colab (только платный тариф)

⚠️ **Только Colab Pro / Pay-As-You-Go с положительным балансом compute units.**
FAQ бесплатного Colab прямо запрещает «running distributed computing workers»;
на платных тарифах это ограничение снято. Этот ноутбук откажется работать,
пока вы не подтвердите платный тариф (секретом или в конфиге).

## Настройка ОДИН раз на аккаунт (потом — просто Run all)

Ноутбук каждый раз открывается из GitHub заново, поэтому правки ячеек не
сохраняются. Вместо правок всё читается из **Colab Secrets** (ключ 🔑 слева) —
добавьте секреты один раз и включайте «Notebook access» при первом запросе:

| Секрет | Значение |
|---|---|
| `GH_TOKEN` | GitHub-токен с правом на Gists — тот же, что в панели мастера |
| `I_USE_PAID_COLAB` | `true` — подтверждение платного тарифа |
| `MODEL_PRESET` | опционально: `minimax_h3` / `sdxl` (пусто = без пресета) |
| `HF_TOKEN` | опционально: gated-модели Hugging Face |
| `GIST_ID` | опционально: обычно НЕ нужен — gist находится сам по GH_TOKEN |

Дальше каждый запуск: **открыть ноутбук → Runtime → Run all**. Воркер сам
опубликует адрес в gist, мастер подхватит его за ~30 секунд.

Рекомендуемый GPU: **A100 (40/80 ГБ)** для MiniMax H3, **L4 (24 ГБ)** для
fp8-видео (Wan) и Flux.

In [ ]:
# ================= КОНФИГ =================
# Вводить ничего не нужно: всё берётся из Colab Secrets (🔑 слева, один раз
# на аккаунт — см. шапку). Поля ниже — ручной запасной путь на одну сессию
# (правки не сохраняются: ноутбук открывается из GitHub заново).
REPO_URL = "https://github.com/Weloyo/ComfyUI-GPU-RAID"
TOKEN = ""                 # пусто = сгенерировать
MODEL_PRESET = ""          # пусто = секрет MODEL_PRESET, иначе без пресета
USE_DRIVE_CACHE = True     # кэш моделей в Google Drive — не перекачивать каждый рантайм
I_USE_PAID_COLAB = False   # подтверждение Pro/PAYG — лучше секретом I_USE_PAID_COLAB=true
GIST_ID = ""               # пусто = секрет GIST_ID или автопоиск по GH_TOKEN
MAX_SESSION_MIN = 0        # самостраховка: погасить рантайм через N минут (0 = выкл)


def secret(name):
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""


HF_TOKEN = secret("HF_TOKEN")
GH_TOKEN = secret("GH_TOKEN")
GIST_ID = (GIST_ID or secret("GIST_ID")).strip()
MODEL_PRESET = (MODEL_PRESET or secret("MODEL_PRESET") or "none").strip().lower()
assert MODEL_PRESET in ("sdxl", "minimax_h3", "none"), f"неизвестный MODEL_PRESET: {MODEL_PRESET}"
if not I_USE_PAID_COLAB:
    I_USE_PAID_COLAB = secret("I_USE_PAID_COLAB").lower() in ("1", "true", "yes", "да")
assert I_USE_PAID_COLAB, (
    "Подтвердите платный Colab (Pro/PAYG): добавьте секрет I_USE_PAID_COLAB "
    "со значением true (🔑 слева, один раз на аккаунт) или поставьте "
    "I_USE_PAID_COLAB = True в этой ячейке. На бесплатном тарифе "
    "распределённые воркеры запрещены правилами.")

# minimax_h3 (~40 ГБ весов): на стандартной Colab RAM тесно — если есть Runtime →
# Change runtime type → High-RAM, включите его. --cache-none экономит RAM всегда.
EXTRA_ARGS = ("--cache-none",) if MODEL_PRESET == "minimax_h3" else ()  # L4/A100: bf16/fp8 работают, force-fp16 не нужен

if not GH_TOKEN:
    print("! нет секрета GH_TOKEN — автоподключение (gist) работать не будет, "
          "строку gpuraid:// придётся копировать вручную")
print(f"config ok · preset={MODEL_PRESET}")

In [ ]:
# ============ ИСХОДНИКИ РАСШИРЕНИЯ ============
import os, sys, subprocess
SRC = "/content/gpu-raid-src"
if not os.path.isdir(SRC):
    assert subprocess.run(["git", "clone", "--depth", "1", REPO_URL, SRC]).returncode == 0,         "git clone не удался — проверьте REPO_URL"
sys.path.insert(0, os.path.join(SRC, "scripts"))
import worker_bootstrap as wb
TOKEN = wb.gen_token(TOKEN)
print("TOKEN:", TOKEN)

# rendezvous-gist мастера находится по одному GH_TOKEN — id вводить не нужно
if not GIST_ID and GH_TOKEN:
    try:
        GIST_ID = wb.discover_gist(GH_TOKEN)
    except Exception as e:
        print(f"! автопоиск гиста не удался: {e}")
    if GIST_ID:
        print("gist найден по токену:", GIST_ID)
    else:
        print("! gist не найден: панель мастера → «Подключения и ключи» → "
              "GitHub → «Создать приватный gist», затем перезапустите ячейку")
subprocess.run(["nvidia-smi", "-L"])

In [ ]:
# ============ (опция) КЭШ МОДЕЛЕЙ В DRIVE ============
COMFY_DIR = "/content/ComfyUI"
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = "/content/drive/MyDrive/gpuraid_models"
    os.makedirs(cache, exist_ok=True)
    # Отдельная плоская папка под hf_preset (напр. minimax_h3): просто копии
    # готовых файлов, БЕЗ вложенных symlink-цепочек huggingface_hub — Drive
    # FUSE-mount ненадёжен именно для них (проверено: крупные файлы по 5-20 ГБ
    # тихо "терялись" при кэшировании через HF_HOME=Drive). Первая сессия
    # качает с HF и сохраняет сюда копию, все следующие копируют отсюда вместо
    # повторного скачивания.
    hf_drive_cache = "/content/drive/MyDrive/gpuraid_hf_cache"
    os.makedirs(hf_drive_cache, exist_ok=True)
    # после установки ComfyUI (следующая ячейка) файлы из кэша прилинкуются:
    def link_drive_models():
        n = 0
        for folder in os.listdir(cache):
            src_dir = os.path.join(cache, folder)
            if not os.path.isdir(src_dir):
                continue
            for f in os.listdir(src_dir):
                n += wb._link(os.path.join(src_dir, f), COMFY_DIR, folder, f)
        print(f"[drive] прилинковано: {n}")
else:
    hf_drive_cache = None
    def link_drive_models():
        pass
print("ok")

In [ ]:
# ============ УСТАНОВКА + ЗАПУСК + ТУННЕЛЬ ============
wb.install_comfy(COMFY_DIR)
link_drive_models()
info = wb.bring_up(
    gpuraid_src=SRC,
    comfy_dir=COMFY_DIR,
    token=TOKEN,
    gpus=(0,),
    base_port=8188,
    extra_args=EXTRA_ARGS,
    use_datasets=False,
    hf_preset=MODEL_PRESET,
    hf_token=HF_TOKEN,
    name_prefix="colab",
    drive_cache_dir=hf_drive_cache,
    gist_id=GIST_ID,
    gh_token=GH_TOKEN,
    max_session_min=MAX_SESSION_MIN,
)

In [ ]:
# ============ WATCHDOG ============
# Следит за процессами и туннелем (умерший туннель перезапустит и перепубликует
# адрес в gist). Завершается сам, когда мастер пришлёт команду остановки
# (режимы Эко/«Сразу гасить») — тогда рантайм освобождается runtime.unassign().
wb.watchdog(info)